# Deepfake Detector — Colab Training
CLIP ViT-L/14 + DoRA + FFT fusion

**Runtime → Change runtime type → A100 GPU**

In [ ]:
# 1. Install dependencies
!pip install -q torch torchvision transformers peft albumentations opencv-python scikit-learn tqdm kagglehub

In [ ]:
# 1b. Mount Google Drive and cd into your project folder
from google.colab import drive
drive.mount('/content/drive')

# Change this to match your folder path in Drive
%cd /content/drive/MyDrive/Colab Notebooks
!ls *.py

In [ ]:
# 2. Upload model.py and train.py
#    Drag both files into the file browser (folder icon on the left sidebar)
#    Then run this cell to verify they're there
import os
needed = ["model.py", "train.py"]
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print(f"Missing files: {missing}")
    print("Drag them into the Files panel on the left sidebar, then re-run this cell.")
else:
    print("All files found!")

In [ ]:
# 3. Download datasets from Kaggle
import kagglehub

print("Downloading Dataset 1: manjilkarki/deepfake-and-real-images...")
ds1_path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")
print(f"  → {ds1_path}")

print("Downloading Dataset 2: xhlulu/140k-real-and-fake-faces...")
ds2_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(f"  → {ds2_path}")

In [ ]:
# 4. Patch dataset paths in train.py for Colab
import re

with open("train.py", "r") as f:
    code = f.read()

code = re.sub(
    r'DATASET_1\s*=\s*Path\(.*?\)',
    f'DATASET_1       = Path("{ds1_path}/Dataset")',
    code
)
code = re.sub(
    r'DATASET_2\s*=\s*Path\(.*?\)',
    f'DATASET_2       = Path("{ds2_path}/real_vs_fake/real-vs-fake")',
    code
)

with open("train.py", "w") as f:
    f.write(code)

print("Patched dataset paths in train.py")

In [ ]:
# 5. Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → A100 GPU")

In [ ]:
# 6. Train!
!python train.py

In [ ]:
# 7. Download trained model
from google.colab import files
import shutil
shutil.make_archive("model", "zip", "model")
files.download("model.zip")